# Stochastic Volatility Deep Dive

A guided tour of the `stochastic-vol-models` library: the Heston characteristic function, the three Fourier pricers, variance simulation, and a calibration round-trip.

Run from the repository root so the package imports resolve.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
from models.heston import HestonModel, HestonParams

params = HestonParams(kappa=2.0, theta=0.04, xi=0.5, rho=-0.7, V0=0.04)
model = HestonModel(params)
print('Feller ratio  :', round(model.feller_ratio, 3))
print('Long-run vol  :', round(model.long_run_vol, 4))
print('Vol-of-vol    :', round(model.vol_of_vol, 4))
print('Var half-life :', round(model.half_life, 3), 'years')

## 1. The characteristic function
Verify the three fundamental properties: $\phi(0)=1$, $|\phi(u)|\le 1$, and the martingale condition $\phi(-i)=F$.

In [ ]:
from characteristic_functions.heston_cf import HestonCharacteristicFunction
cf = HestonCharacteristicFunction()
S0, T, r, q = 100.0, 1.0, 0.02, 0.0
args = (S0, params.V0, T, r, q, params.kappa, params.theta, params.xi, params.rho)
print('phi(0)      =', cf.cf(np.array([0.0]), *args)[0])
u = np.linspace(-200, 200, 4001)
print('max|phi(u)| =', np.abs(cf.cf(u, *args)).max())
print('phi(-i)/F   =', (cf.cf(np.array([-1j]), *args)[0] / (S0*np.exp((r-q)*T))).real)

## 2. Three Fourier pricers agree to < 1 bp of IV

In [ ]:
from fourier_pricing.cos_method import COSMethodPricer
from fourier_pricing.carr_madan import CarrMadanPricer
from fourier_pricing.lewis_method import LewisMethodPricer
from implied_vol.bsm_inversion import BSMInverter
cos, cm, lew, inv = COSMethodPricer(), CarrMadanPricer(), LewisMethodPricer(), BSMInverter()
for K in [80, 100, 120]:
    ivs = [inv.implied_vol(p.price(S0, K, T, r, q, params), S0, K, T, r, q)
           for p in (cos, cm, lew)]
    print(f'K={K}:  COS={ivs[0]:.5f}  CM={ivs[1]:.5f}  Lewis={ivs[2]:.5f}')

## 3. QE simulation — martingale and variance moment

In [ ]:
from simulation.heston_qe import HestonQESimulator
S, V = HestonQESimulator().simulate(S0, params.V0, T, r, q, params, 100000, 50, seed=1)
print('E[S_T] =', round(S[:, -1].mean(), 4), ' forward =', round(S0*np.exp((r-q)*T), 4))
EVT = params.theta + (params.V0 - params.theta)*np.exp(-params.kappa*T)
print('E[V_T] =', round(V[:, -1].mean(), 5), ' theory  =', round(EVT, 5))
print('min variance over all paths =', V.min())

## 4. Calibration round-trip
Generate a surface from known parameters, then recover them.

In [ ]:
from calibration.market_data import MarketIVSurface
from calibration.heston_calibration import HestonCalibrator
true = HestonParams(2.0, 0.04, 0.5, -0.6, 0.045)
strikes, mats = np.array([90, 100, 110.0]), np.array([0.5, 1.0])
surf = MarketIVSurface.generate_heston_surface(true, S0, r, q, strikes, mats)
res = HestonCalibrator(seed=0).calibrate_joint(surf, S0, r, q, 'rmse', de_maxiter=30)
print('true     :', true.as_dict())
print('recovered:', {k: round(v, 4) for k, v in res.params.as_dict().items()})
print('RMSE (bps):', round(res.rmse_bps, 2))